# Emotion Recognition Training Notebook With POSTER

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/codee/1-source-final

/content/drive/.shortcut-targets-by-id/1dHP1zaCyRhJudTFBZ_ivw48q0waIdsjx/1-source-final


## 1. RAF-DB

### 1.0 Preapre data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuvoalok/raf-db-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'raf-db-dataset' dataset.
Path to dataset files: /kaggle/input/raf-db-dataset


In [ ]:
!ls /kaggle/input/raf-db-dataset

DATASET  test_labels.csv  train_labels.csv


### 1.1 Test checkpoint

In [ ]:
!python test.py --dataset rafdb --datapath "{path}" --checkpoint checkpoint-save/rafdb_best.pth -p --poster

Work on GPU:  0
Using RAF-DB path: /kaggle/input/raf-db-dataset/DATASET
load_weight 304
Loading pretrained weights... checkpoint-save/rafdb_best.pth
load_weight 1309
Test set size: 3068
Test accuracy: 0.9143.
Saving confusion matrix to: /content/drive/MyDrive/Group1-23127084-23127250-23127493/code/Confusion_matrix/RAF-DB/acc0.9143.png
Figure(640x480)


### 1.2 Train model

In [ ]:
!python train.py --dataset rafdb --datapath "{path}" --batch_size 64 --modeltype large --epochs 1 --gpu 0 --poster

Work on GPU:  0
Using RAF-DB path: /kaggle/input/raf-db-dataset/DATASET
load_weight 304
Train set size: 12271
Validation set size: 3068
batch_size: 64
Total Parameters: 70.843M
[Epoch 1/1][Train 1/192] loss: 5.8949 acc: 0.1250 lr: 0.000040
[Epoch 1/1][Train 2/192] loss: 5.5332 acc: 0.2734 lr: 0.000040
[Epoch 1/1][Train 3/192] loss: 5.2803 acc: 0.3385 lr: 0.000040
[Epoch 1/1][Train 4/192] loss: 5.2110 acc: 0.3711 lr: 0.000040
[Epoch 1/1][Train 5/192] loss: 5.1644 acc: 0.3875 lr: 0.000040
[Epoch 1/1][Train 6/192] loss: 5.1191 acc: 0.4036 lr: 0.000040
[Epoch 1/1][Train 7/192] loss: 5.0508 acc: 0.4330 lr: 0.000040
[Epoch 1/1][Train 8/192] loss: 5.0534 acc: 0.4375 lr: 0.000040
[Epoch 1/1][Train 9/192] loss: 4.9673 acc: 0.4601 lr: 0.000040
[Epoch 1/1][Train 10/192] loss: 4.9081 acc: 0.4688 lr: 0.000040
[Epoch 1/1][Train 11/192] loss: 4.8719 acc: 0.4702 lr: 0.000040
[Epoch 1/1][Train 12/192] loss: 4.8355 acc: 0.4753 lr: 0.000040
[Epoch 1/1][Train 13/192] loss: 4.8313 acc: 0.4736 lr: 0.000040


### 1.3 Test model

In [ ]:
!python test.py --dataset rafdb --datapath "{path}" --checkpoint checkpoint/best.pth -p --poster

Work on GPU:  0
Using RAF-DB path: /kaggle/input/raf-db-dataset/DATASET
load_weight 304
Loading pretrained weights... checkpoint/best.pth
load_weight 645
Test set size: 3068
Test accuracy: 0.2862.
Saving confusion matrix to: /content/drive/MyDrive/Group1-23127084-23127250-23127493/code/Confusion_matrix/RAF-DB/acc0.2862.png
Figure(640x480)


## 2. AffectNet 7 Classes

### 2.0 Prepare data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mstjebashazida/affectnet")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'affectnet' dataset.
Path to dataset files: /kaggle/input/affectnet


In [ ]:
# Tao dataset AffectNet 7 classes tu AffectNet 8 classes
import os
import shutil
from pathlib import Path

src_root = Path('/kaggle/input/affectnet/archive (3)')
dst_root = Path('/content/drive/MyDrive/POSTER/data/AffectNet')

# Label order cho 7 classes
class_aliases = {
    'neutral': ['neutral'],
    'happy': ['happy'],
    'sad': ['sad'],
    'surprise': ['surprise'],
    'fear': ['fear'],
    'disgust': ['disgust'],
    'anger': ['anger', 'Anger'],
}
class_to_label = {
    'neutral': 0,
    'happy': 1,
    'sad': 2,
    'surprise': 3,
    'fear': 4,
    'disgust': 5,
    'anger': 6,
}

train_src = src_root / 'Train'
valid_src = src_root / 'Test'

train_img_dst = dst_root / 'train_set' / 'images'
valid_img_dst = dst_root / 'valid_set' / 'images'
train_img_dst.mkdir(parents=True, exist_ok=True)
valid_img_dst.mkdir(parents=True, exist_ok=True)

train_ann_path = dst_root / 'train_set' / 'train_annotations_7class.txt'
valid_ann_path = dst_root / 'valid_set' / 'valid_annotations_7class.txt'

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def is_dir_empty(path):
    return not any(path.iterdir())

def link_or_copy(src, dst):
    try:
        if dst.exists():
            return
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def find_class_dir(split_src, aliases):
    for alias in aliases:
        candidate = split_src / alias
        if candidate.exists():
            return candidate
    return None

def build_split(split_src, split_dst, ann_path):
    count = 0
    with ann_path.open('w', encoding='utf-8') as f:
        for cls in sorted(class_to_label.keys()):
            cls_dir = find_class_dir(split_src, class_aliases[cls])
            if cls_dir is None:
                print(f'[WARN] Missing class folder for {cls}: {split_src}')
                continue
            label = class_to_label[cls]
            for p in sorted(cls_dir.iterdir()):
                if p.is_file() and p.suffix.lower() in img_exts:
                    # Prefix de tranh trung ten giua classes
                    new_name = f'{cls}_{p.name}'
                    dst_path = split_dst / new_name
                    link_or_copy(p, dst_path)
                    f.write(f'{new_name} {label}\n')
                    count += 1
    return count

if is_dir_empty(train_img_dst) and is_dir_empty(valid_img_dst):
    train_count = build_split(train_src, train_img_dst, train_ann_path)
    valid_count = build_split(valid_src, valid_img_dst, valid_ann_path)

    print('Done creating AffectNet7 dataset')
    print('Root:', dst_root)
    print('Train images:', train_count)
    print('Valid images:', valid_count)
    print('Train ann:', train_ann_path)
    print('Valid ann:', valid_ann_path)
else:
    print('Skip creating AffectNet7 dataset: destination folders are not empty.')
    print('Train images dir:', train_img_dst)
    print('Valid images dir:', valid_img_dst)

Done creating AffectNet7 dataset
Root: /content/drive/MyDrive/POSTER/data/AffectNet
Train images: 14549
Valid images: 13206
Train ann: /content/drive/MyDrive/POSTER/data/AffectNet/train_set/train_annotations_7class.txt
Valid ann: /content/drive/MyDrive/POSTER/data/AffectNet/valid_set/valid_annotations_7class.txt


### 2.1 Test checkpoint

In [ ]:
!python test_affect.py --dataset affectnet --datapath "/content/drive/MyDrive/POSTER/data/AffectNet" --checkpoint checkpoint/affect_best.pth -p --poster

Work on GPU:  0
Using AffectNet7 path: /content/drive/MyDrive/POSTER/data/AffectNet
load_weight 304
Loading pretrained weights... checkpoint/affect_best.pth
Traceback (most recent call last):
  File "/content/drive/MyDrive/Group1-23127084-23127250-23127493/code/test_affect.py", line 173, in <module>
    test()
  File "/content/drive/MyDrive/Group1-23127084-23127250-23127493/code/test_affect.py", line 115, in test
    checkpoint = torch.load(args.checkpoint, weights_only=False)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in _

### 2.2 Train model

In [ ]:
!python train_affect.py --dataset affectnet --datapath "/content/drive/MyDrive/POSTER/data/AffectNet" --batch_size 64 --modeltype large --epochs 1 --gpu 0 --poster

Work on GPU:  0
load_weight 304
Train set size: 14549
Validation set size: 13206
batch_size: 64
Total Parameters: 70.843M
[Epoch 1/1][Train 1/228] loss: 5.7585 acc: 0.1875 lr: 0.000040
[Epoch 1/1][Train 2/228] loss: 5.6644 acc: 0.2812 lr: 0.000040
[Epoch 1/1][Train 3/228] loss: 5.5862 acc: 0.3229 lr: 0.000040
[Epoch 1/1][Train 4/228] loss: 5.4925 acc: 0.3438 lr: 0.000040
[Epoch 1/1][Train 5/228] loss: 5.3832 acc: 0.3594 lr: 0.000040
[Epoch 1/1][Train 6/228] loss: 5.2808 acc: 0.3750 lr: 0.000040
[Epoch 1/1][Train 7/228] loss: 5.2048 acc: 0.3929 lr: 0.000040
[Epoch 1/1][Train 8/228] loss: 5.1188 acc: 0.4043 lr: 0.000040
[Epoch 1/1][Train 9/228] loss: 5.0747 acc: 0.4132 lr: 0.000040
[Epoch 1/1][Train 10/228] loss: 5.0014 acc: 0.4219 lr: 0.000040
[Epoch 1/1][Train 11/228] loss: 4.9184 acc: 0.4375 lr: 0.000040
[Epoch 1/1][Train 12/228] loss: 4.8750 acc: 0.4427 lr: 0.000040
[Epoch 1/1][Train 13/228] loss: 4.7986 acc: 0.4555 lr: 0.000040
[Epoch 1/1][Train 14/228] loss: 4.7605 acc: 0.4598 lr: 

### 2.3 Test model

In [ ]:
!python test_affect.py --dataset affectnet --datapath "/content/drive/MyDrive/POSTER/data/AffectNet" --checkpoint checkpoint/best.pth -p --poster

Work on GPU:  0
Using AffectNet7 path: /content/drive/MyDrive/POSTER/data/AffectNet
load_weight 304
Loading pretrained weights... checkpoint/best.pth
load_weight 645
Test set size: 13206
Test accuracy: 0.1433.
Saving confusion matrix to: /content/drive/MyDrive/Group1-23127084-23127250-23127493/code/Confusion_matrix/AffectNet7/acc0.1433.png
Figure(640x480)


## 3. AffectNet 8 Classes

### 3.0 Prepare dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mstjebashazida/affectnet")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'affectnet' dataset.
Path to dataset files: /kaggle/input/affectnet


In [ ]:
path = Path(path) / 'archive (3)'

In [ ]:
!ls -lh "{path}"

total 1.5M
-rw-r--r--  1 1000 1000 1.5M Apr 19 02:52 labels.csv
drwxr-sr-x 10 1000 1000    0 Apr 19 02:51 Test
drwxr-sr-x 10 1000 1000    0 Apr 19 02:52 Train


### 3.1 Test checkpoint

In [ ]:
!python test_affect.py --dataset affectnet8class --datapath "{path}" --checkpoint checkpoint/affect8_best.pth -p --poster

Work on GPU:  0
Using AffectNet8 path: /kaggle/input/affectnet/archive (3)
load_weight 304
Loading pretrained weights... checkpoint/affect8_best.pth
Traceback (most recent call last):
  File "/content/drive/MyDrive/Group1-23127084-23127250-23127493/code/test_affect.py", line 173, in <module>
    test()
  File "/content/drive/MyDrive/Group1-23127084-23127250-23127493/code/test_affect.py", line 115, in test
    checkpoint = torch.load(args.checkpoint, weights_only=False)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__


### 3.2 Train model

In [ ]:
!python train_affect.py --dataset affectnet8class --datapath "{path}" --batch_size 64 --modeltype large --epochs 1 --gpu 0 --poster

Work on GPU:  0
Using AffectNet8 path: /kaggle/input/affectnet/archive (3)
load_weight 304
Train set size: 16108
Validation set size: 14518
batch_size: 64
Total Parameters: 70.844M
[Epoch 1/1][Train 1/252] loss: 6.3147 acc: 0.0781 lr: 0.000040
[Epoch 1/1][Train 2/252] loss: 6.3702 acc: 0.1406 lr: 0.000040
[Epoch 1/1][Train 3/252] loss: 6.1432 acc: 0.2083 lr: 0.000040
[Epoch 1/1][Train 4/252] loss: 5.9922 acc: 0.2617 lr: 0.000040
[Epoch 1/1][Train 5/252] loss: 6.0069 acc: 0.2625 lr: 0.000040
[Epoch 1/1][Train 6/252] loss: 5.8829 acc: 0.2995 lr: 0.000040
[Epoch 1/1][Train 7/252] loss: 5.8344 acc: 0.3125 lr: 0.000040
[Epoch 1/1][Train 8/252] loss: 5.7869 acc: 0.3164 lr: 0.000040
[Epoch 1/1][Train 9/252] loss: 5.7295 acc: 0.3247 lr: 0.000040
[Epoch 1/1][Train 10/252] loss: 5.6544 acc: 0.3375 lr: 0.000040
[Epoch 1/1][Train 11/252] loss: 5.5807 acc: 0.3480 lr: 0.000040
[Epoch 1/1][Train 12/252] loss: 5.5372 acc: 0.3555 lr: 0.000040
[Epoch 1/1][Train 13/252] loss: 5.4942 acc: 0.3630 lr: 0.000

### 3.3 Test model

In [ ]:
!python test_affect.py --dataset affectnet8class --datapath "{path}" --checkpoint checkpoint/best.pth -p --poster

Work on GPU:  0
Using AffectNet8 path: /kaggle/input/affectnet/archive (3)
load_weight 304
Loading pretrained weights... checkpoint/best.pth
load_weight 643
Test set size: 14518
Test accuracy: 0.1239.
Saving confusion matrix to: /content/drive/MyDrive/Group1-23127084-23127250-23127493/code/Confusion_matrix/AffectNet_8class/acc0.1239.png
Figure(640x480)


## 4. FERPlus

### 4.0 Prepare dataset

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arnabkumarroy02/ferplus")

print("Path to dataset files:", path)

100%|██████████| 487M/487M [00:02<00:00, 176MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/arnabkumarroy02/ferplus/versions/3


In [ ]:
!ls -F /kaggle/input/ferplus

test/  train/  validation/


### 4.1 Test checkpoint
*Tác giả không cung cấp checkpoint cho dataset FerPlus*

### 4.2 Train model

In [8]:
!python train.py --dataset ferplus --datapath "{path}" --batch_size 64 --modeltype large --epochs 1 --gpu 0 --poster

Work on GPU:  0
FERPlus dataset: mode=train, loaded 66379 images
FERPlus dataset: mode=test, loaded 3573 images
load_weight 304
Train set size: 66379
Validation set size: 3573
batch_size: 64
Total Parameters: 70.844M
[Epoch 1/1][Train 1/1038] loss: 6.3215 acc: 0.1250 lr: 0.000040
[Epoch 1/1][Train 2/1038] loss: 6.2600 acc: 0.1328 lr: 0.000040
[Epoch 1/1][Train 3/1038] loss: 6.3501 acc: 0.1406 lr: 0.000040
[Epoch 1/1][Train 4/1038] loss: 6.3224 acc: 0.1523 lr: 0.000040
[Epoch 1/1][Train 5/1038] loss: 6.3593 acc: 0.1500 lr: 0.000040
[Epoch 1/1][Train 6/1038] loss: 6.3277 acc: 0.1536 lr: 0.000040
[Epoch 1/1][Train 7/1038] loss: 6.2793 acc: 0.1696 lr: 0.000040
[Epoch 1/1][Train 8/1038] loss: 6.2291 acc: 0.1953 lr: 0.000040
[Epoch 1/1][Train 9/1038] loss: 6.1924 acc: 0.2049 lr: 0.000040
[Epoch 1/1][Train 10/1038] loss: 6.1678 acc: 0.2156 lr: 0.000040
[Epoch 1/1][Train 11/1038] loss: 6.1165 acc: 0.2358 lr: 0.000040
[Epoch 1/1][Train 12/1038] loss: 6.0993 acc: 0.2409 lr: 0.000040
[Epoch 1/1][

### 4.3 Test model

In [9]:
!python test.py --dataset ferplus --datapath "{path}" --checkpoint checkpoint/ferplus_poster_latest.pth -p --poster

Work on GPU:  0
FERPlus dataset: mode=test, loaded 3573 images
load_weight 304
Loading pretrained weights... checkpoint/ferplus_poster_latest.pth
load_weight 1309
Test set size: 3573
Test accuracy: 0.8150.
Saving confusion matrix to: /content/drive/.shortcut-targets-by-id/1dHP1zaCyRhJudTFBZ_ivw48q0waIdsjx/1-source-final/Confusion_matrix/FERPlus/acc0.815.png
Figure(640x480)
